In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from gps_ids.results import collapse_variants, normalize_label, synthetic_info


In [ ]:
def get_base_model(model_name):
    return collapse_variants(normalize_label(model_name))


In [3]:
# Set the filenames for the results CSV file.
datasets_to_process = {
    '2D': '2D_Results.csv',
    'UAV Attack': 'UAVATTACK_Results.csv'
}

In [ ]:
for dataset_name, filename in datasets_to_process.items():
    print(f"--- Processing dataset: {dataset_name} ---")

    # Data Loading
    try:
        base_directory = os.path.join('..', '..')
        file_path = os.path.join(base_directory, filename)
        df = pd.read_csv(file_path)
        print(f"Successfully loaded '{filename}'")
    except FileNotFoundError:
        print(f"Error: The file '{filename}' was not found in '{base_directory}'.")
        print("Skipping this dataset.")
        continue
    except Exception as e:
        print(f"An error occurred while reading '{filename}': {e}")
        print("Skipping this dataset.")
        continue

    # Data Preparation
    _info = df['dataset'].map(synthetic_info)
    df = df[_info.notna()].copy()
    if df.empty:
        print(f"No synthetic data found in '{filename}'. Skipping plots.")
        continue
    _info = _info.dropna()
    df['generator'] = [i.generator for i in _info]
    df['size'] = pd.to_numeric([i.size for i in _info])
    df['base_model'] = df['model'].apply(get_base_model)
    plot_df = df.melt(
        id_vars=['base_model', 'model', 'generator', 'size', 'train_time_sec'],
        value_vars=["mcc", "balanced_accuracy", "f1_macro", "g_mean", "accuracy"],
        var_name='Metric',
        value_name='Score'
    )

    # Visualization
    print(f"Generating performance plot for {dataset_name} data...")
    g = sns.catplot(
        data=plot_df,
        x='generator',
        y='Score',
        hue='size',
        col='base_model',
        kind='bar',
        col_wrap=3,
        height=5,
        aspect=1.2,
        legend_out=True,
        palette='viridis',
        errorbar=None,
    )
    g.fig.suptitle(f'Model Performance for {dataset_name} Synthetic Datasets', y=1.03, fontsize=20)
    g.set_axis_labels("Synthetic Data Generator", "Score", fontsize=12)
    g.set_titles("Model: {col_name}", fontsize=14)
    g.add_legend(title='Dataset Size')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    # Save the figure before showing
    perf_plot_path = f"{dataset_name}_performance.png"
    g.fig.savefig(perf_plot_path, dpi=1000, bbox_inches='tight')
    print(f"Saved performance chart to {perf_plot_path}")
    plt.show()

    print(f"Generating training time plot for {dataset_name} data...")
    g_time = sns.catplot(
        data=df,
        x='generator',
        y='train_time_sec',
        hue='size',
        col='base_model',
        col_wrap=3,
        kind='bar',
        palette='magma',
        height=5,
        aspect=1.2,
        legend_out=True,
        sharey=False,
        errorbar=None
    )
    g_time.fig.suptitle(f'Training Time for {dataset_name} Synthetic Datasets', y=1.03, fontsize=20)
    g_time.set_axis_labels("Synthetic Data Generator", "Training Time (sec)", fontsize=12)
    g_time.set_titles("Model: {col_name}", fontsize=14)
    g_time.add_legend(title='Dataset Size')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    time_plot_path = f"{dataset_name}_training_time.png"
    g_time.fig.savefig(time_plot_path, dpi=1000, bbox_inches='tight')
    print(f"Saved training time chart to {time_plot_path}")
    plt.show()